In [1]:
print('Ritu')

Ritu


In [3]:
import os
from langchain.llms import OpenAI
from dotenv import load_dotenv
load_dotenv()
# Set your provider’s API key and base URL
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

# llm = OpenAI(
#     model_name="mistralai/mistral-7b-instruct-v0.2",
#     temperature=0.7,
#     max_tokens=512,
#     base_url="https://api.ridvay.com/v1",  # or your custom endpoint
# )

# resp = llm.invoke("Explain how solar panels work?")
# print(resp)

In [6]:
import os
from langchain_huggingface import ChatHuggingFace


llm = ChatHuggingFace(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    # task="conversational",
    max_new_tokens=256,
    temperature=0.7
)

print(llm.invoke("Explain LangChain in simple terms."))

ValidationError: 1 validation error for ChatHuggingFace
llm
  Field required [type=missing, input_value={'repo_id': 'mistralai/Mi...256, 'temperature': 0.7}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [7]:
import os
from langchain_huggingface import ChatHuggingFace
from langchain_core.messages import HumanMessage


chat = ChatHuggingFace(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    temperature=0.7,
    max_new_tokens=256,
)

response = chat.invoke(
    [HumanMessage(content="Explain LangChain in simple terms.")]
)

print(response.content)

ValidationError: 1 validation error for ChatHuggingFace
llm
  Field required [type=missing, input_value={'repo_id': 'mistralai/Mi..., 'max_new_tokens': 256}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [8]:
import os
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import HumanMessage

# Set token

# Step 1️⃣ Create endpoint LLM
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="conversational",  # IMPORTANT
    max_new_tokens=256,
    temperature=0.7,
)

# Step 2️⃣ Wrap inside ChatHuggingFace
chat = ChatHuggingFace(llm=llm)

# Step 3️⃣ Invoke
response = chat.invoke(
    [HumanMessage(content="Explain LangChain in simple terms.")]
)

print(response.content)

LangChain is a simple yet powerful tool for building **applications powered by AI language models** (like me!). Think of it as a **"chain"** that helps connect different pieces of AI workflows together, so you can create smarter and more useful AI tools.

Here’s the simplest breakdown:

### **1. The Problem: AI Models Are Limited**
Large language models (LLMs) like me are great at:
- Understanding and generating text.
- Answering questions, summarizing, or translating languages.
- Doing basic internet searches (if plugged in).

But alone, we can’t yet:
- Access the latest data (we only know up to October 2023).
- Remember long-term conversations (our memory resets every chat).
- Use external tools like a calculator, database, or real-time search.
- Follow complex logic or multi-step workflows that require organizing information.

### **2. LangChain’s Solution: Chaining Features Together**
LangChain solves this by acting like a **"brain extension"** for AI models. It links different AI 

In [9]:
from langchain_ai21.chat_models import ChatAI21
from langchain.output_parsers import PydanticOutputParser
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage,BaseMessage,HumanMessage,AIMessage,ToolMessage
from langgraph.types import interrupt,Command 
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode,tools_condition
from typing import TypedDict,Annotated, List, Dict, Literal
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg_pool import ConnectionPool
from pydantic import BaseModel
from json_repair import repair_json
import re
from typing import List
from dotenv import load_dotenv
import json
import os
load_dotenv()
DB_URL = os.getenv("PPT_URL")

class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: Dict
    detailed_slides: List[Dict]
    current_slide_index: int
    feedback: str
    topic: str
    action: Literal[ "continue_slide", "update_outline", "update_slide", "complete", '' ]
    tool_caller: Literal["generate_outline","generate_slide_detail"]
class OutlineSlide(BaseModel):
    slide_number: int
    slide_title: str
class OutlineOutput(BaseModel):
    title: str
    total_slides: int
    slides: List[OutlineSlide]
class DetailedPoint(BaseModel):
    key_point: str
    explanation: str
class DetailedSlideOutput(BaseModel):
    slide_number: int
    slide_title: str
    detailed_content: List[DetailedPoint]



# model = ChatAI21(model = 'jamba-mini-2-2026-01')
model = ChatHuggingFace(llm=llm)

searchTool = TavilySearchResults(max_results=3)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)
outline_parser = PydanticOutputParser(pydantic_object= OutlineOutput)
detailed_parser = PydanticOutputParser(pydantic_object= DetailedSlideOutput)
OUTLINE_SYSTEM_PROMPT = SystemMessage(
    content=f"""You are an expert presentation designer.

Your task:
Generate ONLY the presentation title and slide titles.
    
{outline_parser.get_format_instructions()} 

Strict Rules:
- Generate EXACTLY the number of slides requested by the user.
- Do NOT generate key points.
- Do NOT generate slide content.
- Only generate slide_number and slide_title.
- Slide numbers must start from 1 and increment sequentially.
- Ensure logical flow from introduction to conclusion.
- Keep slide titles concise but descriptive.
- Use tools only if factual accuracy is required.
- Return ONLY valid JSON.
- No markdown.
- No explanations.
""")

DETAIL_SYSTEM_PROMPT = SystemMessage(
    content=f"""
You are an expert PowerPoint content generator.

CRITICAL: Return ONLY valid JSON with this EXACT structure:

{detailed_parser.get_format_instructions()}

MANDATORY RULES (violations will fail parsing):
1. EXACTLY 3-5 key_points per slide
2. EACH key_point has EXACTLY ONE "key_point" field AND ONE "explanation" field
3. NO duplicate "explanation" keys - combine into single explanation if needed
4. "explanation" = 2-3 sentences maximum, professional language
5. NO extra fields, NO markdown, NO explanatory text outside JSON
6. Valid JSON only - parser will fail on malformed output

EXAMPLE (follow exactly):
{{
  "slide_number": 1,
  "slide_title": "Your Slide Title",
  "detailed_content": [
    {{
      "key_point": "Single clear bullet point",
      "explanation": "One comprehensive explanation. Two sentences maximum. Professional tone."
    }}
  ]
}}

Generate presentation-ready content for the slide title provided.
"""
)

def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    messages = state["messages"] + [OUTLINE_SYSTEM_PROMPT]
    result = model_with_tools.invoke(messages)
    output = {
        'messages':[result],
        'current_slide_index':0,
        "tool_caller": "generate_outline",
            } 
    if result.content:
        try:
            output['outline'] = outline_parser.parse(repair_json(str(result.content))).model_dump()
            print("output['outline']",output['outline'])
        except json.JSONDecodeError as e:
            print('generate_outline_node',e)
    return output
def generate_slide_detail_node(state: PptState):
    """Step 2: Generate detailed content for current Slide"""
    outline = state['outline']
    current_index = state['current_slide_index']
    detailed_slides = state.get('detailed_slides',[])
    total_slides = len(state.get('outline',{}).get('slides',[]))
    current_slide = outline['slides'][int(current_index)]
    if current_index >= total_slides or current_slide['slide_title'] is None:
        return {"action": "complete"}
    output = {
        "tool_caller": "generate_slide_detail",
        "current_slide_index":current_index
    }
    if state['action'] == "update_slide":
        feedback = state['feedback']
        last_slide = detailed_slides.pop()
        last_outline = outline['slides'][current_index-1]
        output['feedback'] = ''
        output['action'] = ''
        prompt = HumanMessage(
            content=f"""
Update this slide content.

Presentation Title:
{outline['title']}

Outline of the slide:
{last_outline}

Current Slide Content:
{last_slide}

User Feedback:
{feedback}
"""
        )

    else:
        
        
        prompt = HumanMessage(
            content=f"""Generate detailede content for this slide:
Presentation Title: {outline['title']}
Slide Title: {current_slide['slide_title']}
Slide Number: {current_slide['slide_number']}

Provide comprehensive, presentation-ready content."""
    )
    messages = [DETAIL_SYSTEM_PROMPT,prompt]
    if isinstance( state['messages'][-1],ToolMessage):
        messages = state['messages'][-2:]+[DETAIL_SYSTEM_PROMPT,prompt]
    result = model_with_tools.invoke(messages)
    if result.content:
        try:
            detailed_slides.append(detailed_parser.parse(repair_json(str(result.content))).model_dump())
            output['detailed_slides'] = detailed_slides 
            output['current_slide_index'] = current_index +1
            
        except json.JSONDecodeError as e:
            print('generate_slide_detail_node inside',e)
    if output['current_slide_index'] == total_slides:
        output["action"] =  "complete"
    output['messages'] = [result]
    return output
def route_after_tools(state: PptState):
    return state["tool_caller"]
def human_decision(state: PptState):
    decision = interrupt({})
    if decision['action'] == "update_outline":

        return {
            'action': "update_outline",
            "messages":[decision['feedback']]
            }
    elif decision['action'] == 'continue_slide':
        return {'action':'continue_slide'}
    elif decision['action'] == 'update_slide':
        return {'action':'update_slide'}
def route_after_human(state: PptState):
    action = state['action']
    if action == 'update_outline':
        return "generate_outline"
    elif action in ('continue_slide', 'update_slide'):
        return "generate_slide_detail"
    elif action == 'complete':  
        return END
    return END
def build_workflow():
    workflow = StateGraph(PptState)
    workflow.add_node("generate_outline", generate_outline_node)
    workflow.add_node("generate_slide_detail", generate_slide_detail_node)
    workflow.add_node("human_decision", human_decision)
    workflow.add_node("tools", ToolNode(tools))
    workflow.add_edge(START, "generate_outline")
    workflow.add_conditional_edges(
        "generate_outline",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )
    workflow.add_conditional_edges(
        "generate_slide_detail",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )
    workflow.add_conditional_edges(
        "tools",
        route_after_tools,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "generate_slide_detail",
        },
    )
    workflow.add_conditional_edges(
        "human_decision",
        route_after_human,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "generate_slide_detail",
            END: END,
        },
    )
    return workflow
def create_ckeckpointer_and_graph(db_url: str):
    if not db_url:
        raise ValueError('Database Url environment variable not set')
    connection_kwargs = {
            "autocommit": True,
            "prepare_threshold": 0,
        }
    pool = ConnectionPool(
        conninfo=db_url,
            max_size=20,
            kwargs=connection_kwargs,
    )
    checkpointer = PostgresSaver(pool)
    checkpointer.setup()
    workflow = build_workflow()
    graph = workflow.compile(checkpointer=checkpointer)
    return checkpointer, graph


ImportError: cannot import name '_DirectlyInjectedToolArg' from 'langchain_core.tools.base' (c:\Users\kaushal\Desktop\me\AI-Powered PPT Generator\pptenv\Lib\site-packages\langchain_core\tools\base.py)